# Coastal flood step 18: mangrove attribution with dynamic max-distance buffer

This notebook applies a distance-based attribution rule:
- Compute nearest mangrove distance for each asset with positive avoided EAD.
- Set the mangrove buffer distance to the **maximum** of those nearest distances.
- Attribute avoided EAD to mangroves within that dynamic buffer using equal split across nearby patches.

This keeps your existing fixed 1000 m method untouched for side-by-side comparison.


In [ ]:
from pathlib import Path

import pandas
import numpy
import geopandas
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

pandas.set_option('display.max_columns', 200)
pandas.set_option('display.width', 220)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


In [ ]:
# User parameters
SCENARIO = 'minimum'  # 'minimum' or 'maximum'
USE_ONLY_POSITIVE_AVOIDED_FOR_BUFFER = True
ROUND_BUFFER_UP_TO_M = 1.0  # meters

if SCENARIO not in {'minimum', 'maximum'}:
    raise ValueError("SCENARIO must be 'minimum' or 'maximum'.")

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
results_path = base_path / f'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_{SCENARIO}_scenario'
shared_intersections_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections'
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'

output_damage_estimates = results_path / 'damage_estimates'
asset_out = output_damage_estimates / 'coastal_ead_asset_level_usd.csv'
mangrove_path = base_path / 'dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'

for p in [network_csv, asset_out, mangrove_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print(f'SCENARIO: {SCENARIO}')
print(f'asset_out: {asset_out}')
print(f'mangrove_path: {mangrove_path}')


In [ ]:
# Read network metadata and EAD results
network_details = pandas.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_details = network_details[required_cols].drop_duplicates().copy()

asset_ead = pandas.read_csv(asset_out)
print(f'Loaded asset EAD rows: {len(asset_ead):,}')
display(asset_ead.head(10))


In [ ]:
# Build geospatial asset layers from split geometries (same base logic as coastal_flood_07)
network_map_details = network_details[[
    'sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column'
]].drop_duplicates().copy()

map_layers = []
missing_split_files = []

for row in network_map_details.itertuples(index=False):
    split_file = shared_intersections_path / f"{row.asset_gpkg}_splits__coastal_flood_rasters_for_intersections__{row.asset_layer}.geoparquet"
    if not split_file.exists():
        missing_split_files.append(str(split_file))
        continue

    split_geom = geopandas.read_parquet(split_file)
    if split_geom.crs is not None:
        split_geom = split_geom.to_crs('EPSG:3448')

    if row.asset_id_column not in split_geom.columns:
        print(f"Skipping {row.asset_gpkg}_{row.asset_layer}: id column '{row.asset_id_column}' not in split file")
        continue

    split_geom = split_geom[[row.asset_id_column, 'geometry']].copy()
    split_geom = geopandas.GeoDataFrame(split_geom, geometry='geometry', crs=split_geom.crs)

    ead_subset = asset_ead.loc[
        (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
        ['Asset_ID', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']
    ].copy()

    if ead_subset.empty:
        continue

    split_geom['_join_id'] = split_geom[row.asset_id_column].astype(str)
    ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

    merged = split_geom.merge(
        ead_subset[['_join_id', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']],
        on='_join_id',
        how='left'
    )

    merged['Sector'] = row.sector
    merged['Subsector'] = row.asset_description
    merged['Asset'] = row.asset_gpkg
    merged['Layer'] = row.asset_layer
    merged['Asset_ID'] = merged[row.asset_id_column].astype(str)
    merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

    map_layers.append(merged[[
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'geometry'
    ]])

if not map_layers:
    raise ValueError('No map layers could be built from split files.')

sector_avoided_ead_map_layers = geopandas.GeoDataFrame(
    pandas.concat(map_layers, ignore_index=True),
    geometry='geometry',
    crs='EPSG:3448'
)

print(f"Map features loaded: {len(sector_avoided_ead_map_layers):,}")
print('Features by sector:')
display(sector_avoided_ead_map_layers.groupby('Sector', as_index=False).size())

if missing_split_files:
    print('Missing split files (skipped):')
    for p in sorted(set(missing_split_files)):
        print('-', p)


In [ ]:
# Dynamic-buffer attribution: buffer = max nearest distance from positive-avoided assets to mangroves
asset_key_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']

asset_gdf = sector_avoided_ead_map_layers[asset_key_cols + ['Avoided_EAD_USD', 'geometry']].copy()
asset_gdf['Asset_ID'] = asset_gdf['Asset_ID'].astype(str)
asset_gdf = asset_gdf.dissolve(
    by=asset_key_cols,
    as_index=False,
    aggfunc={'Avoided_EAD_USD': 'first'}
)
asset_gdf = geopandas.GeoDataFrame(asset_gdf, geometry='geometry', crs='EPSG:3448')

mangroves = geopandas.read_file(mangrove_path).to_crs('EPSG:3448')
if 'ID' not in mangroves.columns:
    raise KeyError("Expected an 'ID' column in mangrove shapefile.")

mangroves['Mangrove_ID'] = mangroves['ID'].astype(int)
mangrove_base_cols = ['Mangrove_ID']
for c in ['Parish', 'HECTARES', 'TYPE']:
    if c in mangroves.columns:
        mangrove_base_cols.append(c)

if USE_ONLY_POSITIVE_AVOIDED_FOR_BUFFER:
    distance_assets = asset_gdf.loc[asset_gdf['Avoided_EAD_USD'] > 0, asset_key_cols + ['Avoided_EAD_USD', 'geometry']].copy()
else:
    distance_assets = asset_gdf[asset_key_cols + ['Avoided_EAD_USD', 'geometry']].copy()

if distance_assets.empty:
    raise ValueError('No assets available for distance-based buffer selection (check avoided EAD filter).')

nearest_assets = geopandas.sjoin_nearest(
    distance_assets,
    mangroves[['Mangrove_ID', 'geometry']],
    how='left',
    distance_col='nearest_mangrove_distance_m'
)

if nearest_assets['nearest_mangrove_distance_m'].notna().sum() == 0:
    raise ValueError('Could not compute nearest mangrove distances for assets.')

max_nearest_distance_m = float(nearest_assets['nearest_mangrove_distance_m'].max())
if ROUND_BUFFER_UP_TO_M <= 0:
    raise ValueError('ROUND_BUFFER_UP_TO_M must be > 0.')

mangrove_buffer_m = float(numpy.ceil(max_nearest_distance_m / ROUND_BUFFER_UP_TO_M) * ROUND_BUFFER_UP_TO_M)

print(f"Assets used for distance buffer: {len(distance_assets):,}")
print(f"Max nearest-mangrove distance (m): {max_nearest_distance_m:,.2f}")
print(f"Applied dynamic mangrove buffer (m): {mangrove_buffer_m:,.2f}")

print('Nearest-distance diagnostics (m):')
display(nearest_assets['nearest_mangrove_distance_m'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print('Top 10 farthest assets used to set the buffer:')
display(
    nearest_assets[[
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD', 'Mangrove_ID', 'nearest_mangrove_distance_m'
    ]]
    .sort_values('nearest_mangrove_distance_m', ascending=False)
    .head(10)
)

mangrove_buffers = mangroves[mangrove_base_cols + ['geometry']].copy()
mangrove_buffers['geometry'] = mangrove_buffers.geometry.buffer(mangrove_buffer_m)

joined = geopandas.sjoin(
    asset_gdf,
    mangrove_buffers[['Mangrove_ID', 'geometry']],
    how='left',
    predicate='intersects',
)

joined['nearby_mangrove_count'] = joined.groupby(asset_key_cols)['Mangrove_ID'].transform(lambda s: s.notna().sum())
joined['nearby_mangrove_count'] = joined['nearby_mangrove_count'].fillna(0).astype(int)

joined['Avoided_EAD_USD_attributed'] = numpy.where(
    (joined['Mangrove_ID'].notna()) & (joined['nearby_mangrove_count'] > 0),
    joined['Avoided_EAD_USD'] / joined['nearby_mangrove_count'],
    0.0,
)

asset_unique = asset_gdf[asset_key_cols + ['Avoided_EAD_USD']].copy()
total_avoided_usd = float(asset_unique['Avoided_EAD_USD'].sum())

matched_assets = joined.loc[joined['nearby_mangrove_count'] > 0, asset_key_cols].drop_duplicates()
matched_assets['matched'] = 1
asset_match_status = asset_unique.merge(matched_assets, on=asset_key_cols, how='left')
asset_match_status['matched'] = asset_match_status['matched'].fillna(0).astype(int)

attributed_total_usd = float(joined['Avoided_EAD_USD_attributed'].sum())
unattributed_total_usd = float(asset_match_status.loc[asset_match_status['matched'] == 0, 'Avoided_EAD_USD'].sum())

joined_m = joined.dropna(subset=['Mangrove_ID']).copy()
joined_m['Mangrove_ID'] = joined_m['Mangrove_ID'].astype(int)

mangrove_sector_summary = (
    joined_m.groupby(['Mangrove_ID', 'Sector'], as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
    .sort_values(['Mangrove_ID', 'Sector'])
)

mangrove_total_summary = (
    joined_m.groupby('Mangrove_ID', as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
    .rename(columns={'Avoided_EAD_USD_attributed': 'Total_Avoided_EAD_USD_attributed'})
    .sort_values('Total_Avoided_EAD_USD_attributed', ascending=False)
)

mangrove_attribution_map = mangroves[mangrove_base_cols + ['geometry']].merge(
    mangrove_total_summary,
    on='Mangrove_ID',
    how='left'
)
mangrove_attribution_map['Total_Avoided_EAD_USD_attributed'] = mangrove_attribution_map['Total_Avoided_EAD_USD_attributed'].fillna(0.0)

print(f'Total avoided EAD across assets (USD): {total_avoided_usd:,.2f}')
print(f'Total attributed within dynamic buffer (USD): {attributed_total_usd:,.2f}')
print(f'Total not attributed (outside dynamic buffer) (USD): {unattributed_total_usd:,.2f}')
if abs(total_avoided_usd) > 0:
    print(f'Fraction attributed: {100 * attributed_total_usd / total_avoided_usd:,.1f}%')
else:
    print('Fraction attributed: not defined (total avoided EAD is zero).')


In [ ]:
# Save outputs
out_dir = results_path / 'damage_estimates' / 'mangrove_attribution_dynamic_buffer'
out_dir.mkdir(parents=True, exist_ok=True)

buffer_label = f"{int(round(mangrove_buffer_m))}m"
method_label = f"dynamic_max_nn_{buffer_label}"

nearest_out = nearest_assets[[
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
    'Avoided_EAD_USD', 'Mangrove_ID', 'nearest_mangrove_distance_m'
]].copy()

joined_out = joined[[
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD',
    'Mangrove_ID', 'nearby_mangrove_count', 'Avoided_EAD_USD_attributed'
]].copy()

summary_out = pandas.DataFrame([
    {
        'Scenario': SCENARIO,
        'Use_Positive_Avoided_For_Buffer': USE_ONLY_POSITIVE_AVOIDED_FOR_BUFFER,
        'Max_Nearest_Distance_m': max_nearest_distance_m,
        'Applied_Buffer_m': mangrove_buffer_m,
        'Total_Avoided_EAD_USD': total_avoided_usd,
        'Attributed_Total_USD': attributed_total_usd,
        'Unattributed_Total_USD': unattributed_total_usd,
        'Fraction_Attributed_pct': (100.0 * attributed_total_usd / total_avoided_usd) if abs(total_avoided_usd) > 0 else numpy.nan,
    }
])

nearest_out.to_csv(out_dir / f'asset_nearest_mangrove_distances_{method_label}.csv', index=False)
joined_out.to_csv(out_dir / f'asset_to_nearby_mangrove_attribution_{method_label}.csv', index=False)
mangrove_sector_summary.to_csv(out_dir / f'mangrove_attribution_by_sector_{method_label}.csv', index=False)
mangrove_total_summary.to_csv(out_dir / f'mangrove_attribution_total_{method_label}.csv', index=False)
summary_out.to_csv(out_dir / f'run_summary_{method_label}.csv', index=False)
mangrove_attribution_map.to_file(out_dir / f'mangrove_attribution_total_{method_label}.gpkg', driver='GPKG')

print(f'Saved outputs to: {out_dir}')
print(f'Method label: {method_label}')
print('Top 10 mangroves by attributed avoided EAD (USD):')
display(mangrove_total_summary.head(10))


In [ ]:
# Optional comparison against existing fixed 1000 m step-07 outputs
fixed_out_dir = results_path / 'damage_estimates' / 'mangrove_attribution'
fixed_total_csv = fixed_out_dir / 'mangrove_attribution_total_1000m.csv'

if fixed_total_csv.exists():
    fixed_total = pandas.read_csv(fixed_total_csv)
    fixed_attributed_total_usd = float(fixed_total['Total_Avoided_EAD_USD_attributed'].sum())
    fixed_unattributed_total_usd = total_avoided_usd - fixed_attributed_total_usd

    compare = pandas.DataFrame([
        {
            'Method': 'Fixed buffer 1000m',
            'Buffer_m': 1000.0,
            'Attributed_Total_USD': fixed_attributed_total_usd,
            'Unattributed_Total_USD': fixed_unattributed_total_usd,
            'Fraction_Attributed_pct': (100.0 * fixed_attributed_total_usd / total_avoided_usd) if abs(total_avoided_usd) > 0 else numpy.nan,
        },
        {
            'Method': 'Dynamic buffer (max nearest distance)',
            'Buffer_m': mangrove_buffer_m,
            'Attributed_Total_USD': attributed_total_usd,
            'Unattributed_Total_USD': unattributed_total_usd,
            'Fraction_Attributed_pct': (100.0 * attributed_total_usd / total_avoided_usd) if abs(total_avoided_usd) > 0 else numpy.nan,
        },
    ])

    print(f'Loaded baseline: {fixed_total_csv}')
    display(compare)
else:
    print(f'No baseline fixed 1000 m file found at: {fixed_total_csv}')
    print('Run coastal_flood_07 first if you want direct side-by-side method comparison.')


In [ ]:
# Map mangrove-level attributed avoided EAD (USD) for the dynamic-buffer result
if mangrove_attribution_map.crs is None or str(mangrove_attribution_map.crs).upper() != 'EPSG:3448':
    mangrove_attribution_map = mangrove_attribution_map.to_crs('EPSG:3448')

jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs('EPSG:3448')

value_col = 'Total_Avoided_EAD_USD_attributed'
vals = mangrove_attribution_map[value_col].fillna(0.0)
abs_vals = vals.abs()
true_max_abs = float(abs_vals.max())

display_quantile = 0.995
display_cap = float(abs_vals.quantile(display_quantile))
if display_cap <= 0:
    display_cap = true_max_abs if true_max_abs > 0 else 1.0

mangrove_plot = mangrove_attribution_map.copy()
mangrove_plot['_plot_val'] = vals.clip(-display_cap, display_cap)

cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)
norm = TwoSlopeNorm(vmin=-display_cap, vcenter=0.0, vmax=display_cap)

fig, ax = plt.subplots(figsize=(10.5, 9.0))
ax.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=ax, color='#c7c7c7', linewidth=0.35, zorder=1)

mangrove_plot.plot(
    ax=ax,
    column='_plot_val',
    cmap=cmap,
    norm=norm,
    linewidth=0.25,
    edgecolor='#6f6f6f',
    alpha=0.95,
    zorder=2,
)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
cbar.set_label(
    f"Attributed avoided EAD (USD), clipped at q={display_quantile:.3f} | Red=increase, White=no change, Green=avoided",
    rotation=90,
)

ax.set_title(
    f"FoN mangroves: dynamic-buffer attributed avoided EAD (USD) | buffer={mangrove_buffer_m:,.1f} m | true max abs={true_max_abs:,.2f}",
    fontsize=12,
)
ax.set_axis_off()
plt.tight_layout()

out_png = out_dir / f'mangrove_attribution_map_usd_{method_label}_q{int(display_quantile * 1000)}.png'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print(f'Saved: {out_png}')
plt.show()
